In [1]:
import sys
import os 

project_root = "/Users/janikwahrheit/Library/CloudStorage/OneDrive-Persönlich/01_Studium/01_Bachelor/Bachelorarbeit/Code"

sys.path.append(project_root)

<h1> Training Pipeline </h1>

In [2]:
from utils.analytics import eval_fit_methods
import networks
import estimator.stable_estimators as se
from scipy.stats import levy_stable
import matplotlib.pyplot as plt 
import seaborn as sns 
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import pandas as pd
import pickle
import numpy as np
import torch
import torch.nn as nn
import utils.optimreg as optimreg
from data.dataloaders import get_mnist_loaders, get_cifar_loaders

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

<h2> Configs </h2>

In [4]:
RUNS = 10
# REGULARIZERS = [None, "ridge", "lasso", "hill", "parabolic_hill", "parabolic_hill_spec_layers0", "parabolic_hill_spec_layers1", "parabolic_hill_spec_layers2"]
REGULARIZERS = ["xiao", "decay"]
# REGULARIZERS = [None, "ridge", "lasso", "hill", "parabolic_hill", "parabolic_hill_spec_layers0", "parabolic_hill_spec_layers1", "parabolic_hill_spec_layers2", "xiao", "decay"]
LAMBDA_SEARCH = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]

ARCHITECTURES = {
    "fc3_mnist": [784, 256, 256, 256, 10],
    "fc3_cifar": [32*32*3, 256, 256, 256, 10],
    "fc10_mnist": [784] + [256]*10 + [10],
    "fc10_cifar": [32*32*3] + [256]*10 + [10]
}

EPOCHS = {
    "fc3_mnist": 10,
    "fc10_mnist": 200,
    "fc3_cifar": 50,
    "fc10_cifar": 200
}

TUNING_EPOCHS = {
    "fc3_mnist": 5,   
    "fc10_mnist": 20,  
    "fc3_cifar": 10,
    "fc10_cifar": 20
}


<h2> Hyperparameter Tuning </h2> 

In [ ]:
def tune_lambda(dataset_name, arch_key, optimizer_name, train_loader, val_loader, reg):
    """
    Führe Hyperparameter-Tuning für lambda durch (nur bei Regularizer != None)
    """
    architecture = ARCHITECTURES[arch_key]

    if reg is None:
        return 0.0  # Kein Tuning für None

    best_acc = -1.0
    best_lambda = LAMBDA_SEARCH[0]

    for lam in LAMBDA_SEARCH:
        model = networks.FCNet(
            layer_sizes=architecture,
            activation='relu',
            weight_init='gaussian'
        ).to(device)

        optimizer = optimreg.get_optimizer(model, optimizer_name=optimizer_name, lr=1e-3)
        epochs = TUNING_EPOCHS[arch_key]

        networks.train(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            epochs=epochs,
            model_name=f"tuning_{arch_key}_{reg}_{lam}_{optimizer_name}_{dataset_name}",
            logging=False,
            run=0,  # nur 1 Run
            regularizer=reg,
            lambda_reg=lam,
            architecture=architecture
        )

        acc = networks.evaluate(model, val_loader)
        if acc > best_acc:
            best_acc = acc
            best_lambda = lam

    print(f"Best lambda for {arch_key} | {reg} | {optimizer_name}: {best_lambda}")
    return best_lambda

<h2> Training Runs </h2>

In [6]:
def run_experiment(dataset_name, arch_key, optimizer_name, train_loader, val_loader, test_loader):
    architecture = ARCHITECTURES[arch_key]

    test_accuracy_log = {}

    for reg in REGULARIZERS:
        if (optimizer_name == "sgd") & (reg == "xiao"): 
            continue
        # Hyperparameter-Tuning direkt vor jedem Regularizer
        lam = tune_lambda(dataset_name, arch_key, optimizer_name, train_loader, val_loader, reg)

        accuracies = []

        for run in range(RUNS):
            print("\n==============================================")
            print(f"Dataset: {dataset_name}")
            print(f"Model: {arch_key}")
            print(f"Regularizer: {reg}")
            print(f"Lambda: {lam}")
            print(f"Optimizer: {optimizer_name}")
            print(f"Run: {run}")
            print("==============================================\n")

            model = networks.FCNet(
                layer_sizes=architecture,
                activation='relu',
                weight_init='gaussian'
            ).to(device)

            optimizer = optimreg.get_optimizer(model, optimizer_name=optimizer_name, lr=1e-3)
            epochs = EPOCHS[arch_key]
            name = f"{arch_key}_{reg}_{optimizer_name}_{dataset_name}"

            networks.train(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                optimizer=optimizer,
                epochs=epochs,
                model_name=name,
                logging=True,
                run=run,
                regularizer=reg,
                lambda_reg=lam,
                architecture=architecture
            )

            acc = networks.evaluate(model, test_loader)
            accuracies.append(acc)
        
        test_accuracy_log[name] = accuracies

        print("\n================================")
        print(f"Finished: {arch_key} | Reg: {reg} | λ={lam} | Optimizer: {optimizer_name}")
        print("Accuracies:", accuracies)
        print("Mean:", np.mean(accuracies))
        print("Std:", np.std(accuracies))
        print("================================\n")
    
    with open(r"/Users/janikwahrheit/Library/CloudStorage/OneDrive-Persönlich/01_Studium/01_Bachelor/Bachelorarbeit/Code/data/training_results/cifarTestAccuracies.pkl", "wb") as f:
        pickle.dump(test_accuracy_log, f)





In [7]:
#OPTIMIZERS = ["sgd", "adam"]
OPTIMIZERS = ["sgd", "adam"]
BATCH_SIZE = 32


In [9]:


# MNIST
train_loader, val_loader, test_loader = get_mnist_loaders(batch_size=BATCH_SIZE)
for opt in OPTIMIZERS:
    print(f"\n===== Running MNIST FC3 | Optimizer: {opt} =====")
    run_experiment("mnist", "fc3_mnist", opt, train_loader, val_loader, test_loader)

    print(f"\n===== Running MNIST FC10 | Optimizer: {opt} =====")
    run_experiment("mnist", "fc10_mnist", opt, train_loader, val_loader, test_loader)



===== Running MNIST FC3 | Optimizer: sgd =====


TypeError: unsupported operand type(s) for &: 'str' and 'str'

In [8]:

# CIFAR
train_loader, val_loader, test_loader = get_cifar_loaders(batch_size=BATCH_SIZE)
for opt in OPTIMIZERS:
    print(f"\n===== Running CIFAR FC3 | Optimizer: {opt} =====")
    run_experiment("cifar", "fc3_cifar", opt, train_loader, val_loader, test_loader)

    # print(f"\n===== Running CIFAR FC10 | Optimizer: {opt} =====")
    # run_experiment("cifar", "fc10_cifar", opt, train_loader, val_loader, test_loader)


===== Running CIFAR FC3 | Optimizer: sgd =====
Epoch 1/10 | Train Loss: 6.4609 | Train Acc: 20.36% | Val Loss: 2.1882 | Val Acc: 22.84%
Epoch 2/10 | Train Loss: 3.5447 | Train Acc: 26.35% | Val Loss: 2.1002 | Val Acc: 26.67%
Epoch 3/10 | Train Loss: 2.8303 | Train Acc: 27.85% | Val Loss: 2.0400 | Val Acc: 28.89%
Epoch 4/10 | Train Loss: 2.5189 | Train Acc: 29.85% | Val Loss: 2.0017 | Val Acc: 27.66%
Epoch 5/10 | Train Loss: 2.3426 | Train Acc: 30.64% | Val Loss: 1.9673 | Val Acc: 30.44%
Epoch 6/10 | Train Loss: 2.2330 | Train Acc: 31.60% | Val Loss: 1.9423 | Val Acc: 31.36%
Epoch 7/10 | Train Loss: 2.1531 | Train Acc: 32.47% | Val Loss: 1.9241 | Val Acc: 32.26%
Epoch 8/10 | Train Loss: 2.0973 | Train Acc: 33.16% | Val Loss: 1.9064 | Val Acc: 32.30%
Epoch 9/10 | Train Loss: 2.0534 | Train Acc: 33.86% | Val Loss: 1.8964 | Val Acc: 32.36%
Epoch 10/10 | Train Loss: 2.0187 | Train Acc: 34.60% | Val Loss: 1.8778 | Val Acc: 33.69%
Test Accuracy: 33.69%
Epoch 1/10 | Train Loss: 6.5292 | Train

KeyboardInterrupt: 